In [3]:
from transformers import AutoTokenizer, AutoModel
def inspect_sentence(text,tokenizer):
    encoded = tokenizer(text,truncation=True,padding = True, return_tensors="pt")
    return encoded
def inspect_model(encoded,model):
    outputs = model(**encoded)
    print(f"outputs : {outputs[0][0][:10]}")
    print(f"last_hidden_state_shape : {outputs.last_hidden_state.shape}")
    print(f"outputs Type : {type(outputs)}")

In [4]:
checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
text = "I love learning Transformers!"
inspect_model(inspect_sentence(text,tokenizer),model)

outputs : tensor([[ 0.6031,  0.0210,  0.1191,  ..., -0.0704,  0.3448,  0.0084],
        [ 0.3324, -0.0397,  0.2901,  ...,  0.1657,  0.0360,  0.4170],
        [ 0.3275,  0.2188, -0.3740,  ...,  0.5549, -0.4262,  0.0958],
        ...,
        [ 0.2512,  0.3698,  0.1805,  ...,  0.2003,  0.1382, -0.1633],
        [ 0.9154,  0.0804,  0.6942,  ..., -0.0177, -0.0951, -0.1033],
        [ 0.6660,  0.7473,  0.1796,  ...,  0.0396,  0.6812, -0.6343]],
       grad_fn=<SliceBackward0>)
last_hidden_state_shape : torch.Size([1, 7, 768])
outputs Type : <class 'transformers.modeling_outputs.BaseModelOutputWithPoolingAndCrossAttentions'>


In [6]:
embedding_layer = model.embeddings.word_embeddings
encoded = inspect_sentence(text,tokenizer)
input_ids = encoded["input_ids"]
embeddings = model.embeddings(input_ids=input_ids)

print(embeddings.shape) # (Batch Size,Sequence Length, Feature Vector)

torch.Size([1, 7, 768])


In [7]:
import torch
Q_weights = torch.rand(768, 768)
K_weights = torch.rand(768, 768)
V_weights = torch.rand(768, 768)

In [9]:
def get_Q(embeddings,Q_weights):
    Q = embeddings @ Q_weights
    return Q

def get_K(embeddings,K_weights):
    K = embeddings @ K_weights
    return K

def get_V(embeddings,V_weights):
    V = embeddings @ V_weights
    return V



In [18]:
import numpy as np
import torch
import torch.nn as nn

class SelfAttention():
    def __init__ (self,embeddings):
        self.embeddings = embeddings
        self.d_model = embeddings.shape[-1]
        self.d_k = 768
        self.Wq = torch.rand(self.d_model,self.d_model)
        self.Wk = torch.rand(self.d_model,self.d_model)
        self.Wv = torch.rand(self.d_model,self.d_model)

    def get_Q(self):
        Q = self.embeddings @ self.Wq
        return Q

    def get_K(self):
        K = self.embeddings @ self.Wk
        return K

    def get_V(self):
        V = self.embeddings @ self.Wv
        return V

    def AttentionScore(self,Q, K, V):
        Score = Q @ K.transpose(-2,-1) / np.sqrt(self.d_k)
        return Score

    def forward(self):
        Q = self.get_Q()
        K = self.get_K()
        V = self.get_V()
        m = nn.Softmax(dim=-1)
        attention_weights = m(self.AttentionScore(Q, K, V))
        output = attention_weights @ V
        return output, attention_weights

In [19]:
attention = SelfAttention(embeddings)

output, weights = attention.forward()

print(output.shape)
print(weights.shape)

torch.Size([1, 7, 768])
torch.Size([1, 7, 7])
